In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Conv2d, BatchNorm2d, SiLU, Identity, Module, Sequential
import random

In [60]:
class Conv(Sequential):
    def __init__(self, cin, cout, k=3, norm=True, act="silu"):
        super().__init__(
        BatchNorm2d(cin) if norm else Identity(),
        Conv2d(cin, cout, k, padding="same"),
        {"none": Identity(), "silu":SiLU()}[act]
        )
class DoubleConv(Sequential):
    def __init__(self, cin, cout, k=3, norm=True, act="silu"):
        super().__init__(
            Conv(cin, cout, k, norm, act),
            Conv(cout, cout, k, norm, act)
        )

In [61]:
class UNET(Module):
    def __init__(self, c):
        super().__init__()
        self.d1 = DoubleConv(2, c)
        self.d2 = DoubleConv(c, c*2)
        self.d3 = DoubleConv(c*2, c*4)
        self.d4 = DoubleConv(c*4, c*8)

        self.bn = DoubleConv(c*8, c*8)

        self.u4 = DoubleConv(c*16, c*4)
        self.u3 = DoubleConv(c*8, c*2)
        self.u2 = DoubleConv(c*4, c)
        self.u1 = DoubleConv(c*2, c)
        self.u0 = DoubleConv(c, 1)

    def forward(self, x):
        x = F.pad(x, [2,2,2,2])
        x1 = F.max_pool2d(self.d1(x), 2, 2)
        x2 = F.max_pool2d(self.d2(x1), 2, 2)
        x3 = F.max_pool2d(self.d3(x2), 2, 2)
        x4 = F.max_pool2d(self.d4(x3), 2, 2)

        x = self.bn(x4)

        x = F.interpolate(self.u4(torch.cat([x4, x], dim=1)), scale_factor=2)
        x = F.interpolate(self.u3(torch.cat([x3, x], dim=1)), scale_factor=2)
        x = F.interpolate(self.u2(torch.cat([x2, x], dim=1)), scale_factor=2)
        x = F.interpolate(self.u1(torch.cat([x1, x], dim=1)), scale_factor=2)
        x = self.u0(x)
        x = F.pad(x, [-2,-2,-2,-2])
        return x

In [62]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

def get_mnist_dataloaders(batch_size=64):
    # 1. Download and load the training data
    train_dataset = datasets.MNIST(
        root='./data',
        train=True,
        download=True,
        transform=ToTensor()  # Converts images to PIL format/numpy to torch.FloatTensor and scales pixels to [0.0, 1.0]
    )

    # 2. Download and load the test data
    test_dataset = datasets.MNIST(
        root='./data',
        train=False,
        download=True,
        transform=ToTensor()
    )

    # 3. Create DataLoaders
    train_loader = DataLoader(
        dataset=train_dataset, 
        batch_size=batch_size, 
        shuffle=True,          # Shuffle training data to prevent overfitting
        num_workers=2,         # Speeds up data loading by using subprocesses
        pin_memory=True        # Speeds up data transfer to GPU if you are using one
    )

    test_loader = DataLoader(
        dataset=test_dataset, 
        batch_size=batch_size, 
        shuffle=False,         # No need to shuffle test data
        num_workers=2,
        pin_memory=True
    )

    return train_loader, test_loader

# Example usage:
if __name__ == "__main__":
    train_loader, test_loader = get_mnist_dataloaders(batch_size=64)
    
    # Quick sanity check: grab one batch
    images, labels = next(iter(train_loader))
    print(f"Feature batch shape: {images.shape}")  # Should be [64, 1, 28, 28] -> [batch, channel, height, width]
    print(f"Labels batch shape: {labels.shape}")    # Should be [64]

Feature batch shape: torch.Size([64, 1, 28, 28])
Labels batch shape: torch.Size([64])


In [78]:
unet = UNET(16).cuda()

def train_epoch(unet, loader, optimizer, epoch):
    epoch_loss = 0.0
    for batch, _ in loader:
        optimizer.zero_grad()
        eps=torch.randn(batch.shape)
        t=torch.full(batch.shape, random.random())
        z = batch * (torch.ones(batch.shape)-t) + eps * t
        y = eps-batch
        y=y.cuda()
        zt = torch.cat((z, t), dim=1).cuda()
        y_hat = unet(zt).cuda()
        loss = F.mse_loss(y, y_hat)
        loss.backward()
        epoch_loss+=loss.item()
        optimizer.step()

    print(f"epoch: {epoch}   loss: {epoch_loss}")
    return unet

lr=6.7*1e-3
optim = torch.optim.Adam(unet.parameters(), lr=lr)
for epoch in range(100):
    unet = train_epoch(unet, train_loader, optim, epoch)

epoch: 0   loss: 633.2971062064171
epoch: 1   loss: 557.8862961530685
epoch: 2   loss: 549.2121128439903
epoch: 3   loss: 543.5919699072838
epoch: 4   loss: 539.5391097068787
epoch: 5   loss: 543.0678577423096
epoch: 6   loss: 536.1281314492226
epoch: 7   loss: 531.1932579874992
epoch: 8   loss: 529.3494651913643
epoch: 9   loss: 527.0562766194344
epoch: 10   loss: 525.2578234672546
epoch: 11   loss: 522.7003882527351
epoch: 12   loss: 519.2184748053551
epoch: 13   loss: 523.6449144482613
epoch: 14   loss: 521.7458117008209
epoch: 15   loss: 518.6218672990799
epoch: 16   loss: 516.4589115381241
epoch: 17   loss: 520.0653839111328
epoch: 18   loss: 518.4631901979446
epoch: 19   loss: 517.9901741743088
epoch: 20   loss: 516.7248960137367
epoch: 21   loss: 516.6820608377457
epoch: 22   loss: 517.0760786533356
epoch: 23   loss: 514.669153213501
epoch: 24   loss: 515.8974545001984
epoch: 25   loss: 515.8092722892761
epoch: 26   loss: 515.1358489394188
epoch: 27   loss: 514.5213366746902
epo

In [81]:
save_path = f"model.pth"
torch.save(unet.state_dict(), save_path)

print(f"Model successfully saved to {save_path}")


Model successfully saved to model.pth
